# Profile one microscope image

Reproduce one recorded setting from `microscope_calibration_40k_60k`, using the production renderer, 512² coherent beamlets, double precision, and the recorded pixel sampling. The default is the first 4 µm aperture image (200 kV, nominal 40,000×, first positive X shift).

Run only after the experiment has finished and the GPU is idle. The first cell checks this **before importing JAX**. Idle notebook kernels may retain GPU memory; they are neither interrupted nor shut down. A GPU activity log records other processes during profiling.

Compilation and warm-up are excluded from a synchronized, single-image production timing. A second single pass, separated at stage boundaries, attributes the cost and checks agreement with the production image. This diagnostic pass has extra synchronization and cannot preserve cross-stage compiler fusion. A separate warmed production pass records a JAX GPU trace. No experiment files or library code are changed.

**Acquisition status at preparation:** the original 256 × 256 image sweep finished, failed the image/moment audit, and entered 8 × 8 subpixel refinement. The executor then exited with `KeyboardInterrupt` after two refined pairs. The last saved numerical-check file therefore still describes the initial 4 × 4 sweep; the notebook rereads the saved configuration when execution starts. Other GPU jobs must finish before it runs.

In [1]:
from pathlib import Path
import os, json, time, subprocess, atexit, csv, hashlib
from datetime import datetime, timezone

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
EXPERIMENT = ROOT / "examples/generated/microscope_calibration_40k_60k"
SETTING_INDEX = 0
APERTURE_INDEX = 0
GAUSSIANS_ACROSS = 512
INITIAL_SAMPLES_PER_PIXEL = 4
GPU_INDEX = 0

def smi(*args):
    return subprocess.check_output(["nvidia-smi", *args], text=True).strip()

# These are the experiment executor and kernel observed when this notebook was made.
# Match the command as well as PID to avoid blocking on a later reused PID.
for pid, fragment in [(3773719, "run_new_calibration.py"), (3773724, "tmpssb2ptdq.json")]:
    path = Path(f"/proc/{pid}/cmdline")
    if path.exists() and fragment in path.read_text():
        raise RuntimeError(f"The current experiment is still running (PID {pid}). Wait for completion.")

idle_samples = []
for _ in range(4):
    value = int(smi("-i", str(GPU_INDEX), "--query-gpu=utilization.gpu", "--format=csv,noheader,nounits"))
    idle_samples.append(value)
    if value != 0:
        raise RuntimeError(f"GPU is busy ({value}%); rerun once it is idle.")
    time.sleep(5)
print("GPU idle samples (%):", idle_samples)
print(smi("--query-compute-apps=pid,process_name,used_memory", "--format=csv"))

os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_INDEX)
os.environ["JAX_PLATFORMS"] = "cuda"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
RUN_DIR = ROOT / "examples/generated/microscope_single_pass_profile" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR.mkdir(parents=True)
print("Profile artifacts:", RUN_DIR)
monitor = subprocess.Popen(["nvidia-smi", "pmon", "-i", str(GPU_INDEX), "-s", "um", "-d", "1"],
                           stdout=(RUN_DIR / "gpu_activity.log").open("w"), stderr=subprocess.STDOUT)
atexit.register(lambda: monitor.terminate() if monitor.poll() is None else None)

GPU idle samples (%): [0, 0, 0, 0]
pid, process_name, used_gpu_memory [MiB]
3739871, /Users/landers/TemGymCore/.venv/bin/python, 878 MiB
Profile artifacts: /Users/landers/TemGymCore/examples/generated/microscope_single_pass_profile/20260909T130542Z


<function __main__.<lambda>()>

In [2]:
import numpy as np
import jax
import jax.numpy as jnp
from dataclasses import replace
from IPython.display import display, Markdown
from temgym_core.calibration.gaussian import aperture_wave, make_gaussian_renderer
from temgym_core.calibration.apertures import effective_parameters, aperture_centres
from temgym_core.calibration.condenser_model import condenser_with_steering
from temgym_core.calibration.projector_model import at_voltage, projector_components
from temgym_core.components import Detector
from temgym_core.gaussian import FreeSpacePropagator
from temgym_core.run import run_to_end, column_matrix
from temgym_core.evaluate import evaluate_gaussians, gpu_kernel_available, _prepare_gaussian_params, _build_spatial_bins
from jax.scipy.special import ndtr

jax.config.update("jax_enable_x64", True)
assert jax.default_backend() == "gpu"
assert gpu_kernel_available(), "Production spatial/Pallas GPU backend is required."
print("JAX:", jax.__version__, "Devices:", jax.devices(), "PID:", os.getpid())

spec = json.loads((EXPERIMENT / "experiment.json").read_text())
truth = json.loads((EXPERIMENT / "scoring_truth.json").read_text())
with np.load(EXPERIMENT / "settings.npz") as data:
    settings = dict(data)
checks_path = EXPERIMENT / "numerical_checks.json"
checks = json.loads(checks_path.read_text()) if checks_path.exists() else {}
sampling = checks.get("samples_per_pixel_per_pair", [INITIAL_SAMPLES_PER_PIXEL] * len(settings["voltage_index"]))[SETTING_INDEX]
sampling = int(sampling)
vi = int(settings["voltage_index"][SETTING_INDEX])
voltage = spec["voltages_v"][vi]
diameter = spec["aperture_diameters_um"][APERTURE_INDEX]
shape = tuple(spec["shape_yx"])
pitch = spec["pixel_pitch_m"]
padding = 8
metadata = {
    "setting_index": SETTING_INDEX, "aperture_diameter_um": diameter,
    "voltage_v": voltage, "nominal_magnification": float(settings["nominal_magnification"][SETTING_INDEX]),
    "gaussians_across": GAUSSIANS_ACROSS, "gaussians": GAUSSIANS_ACROSS**2,
    "samples_per_pixel_axis": sampling, "initial_samples_per_pixel_axis": INITIAL_SAMPLES_PER_PIXEL,
    "detector_shape": shape, "field_shape": [(v + 2*padding)*sampling for v in shape],
    "jax_version": jax.__version__, "devices": str(jax.devices()), "profile_pid": os.getpid(),
    "idle_samples_percent": idle_samples,
    "source_sha256": {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
        for p in (ROOT / "src/temgym_core").rglob("*.py")},
    "input_sha256": {name: hashlib.sha256((EXPERIMENT / name).read_bytes()).hexdigest()
        for name in ["experiment.json", "settings.npz", "scoring_truth.json"]},
}
metadata["dense_gaussian_sample_pairs"] = int(metadata["gaussians"] * np.prod(metadata["field_shape"]))
print(json.dumps({k: v for k, v in metadata.items() if not k.endswith("sha256")}, indent=2))
timings = {}
def timed(name, fn, *args):
    start = time.perf_counter()
    result = fn(*args)
    jax.block_until_ready(result)
    timings[name] = time.perf_counter() - start
    print(f"{name}: {timings[name]:.6f} s", flush=True)
    return result

JAX: 0.10.2 Devices: [CudaDevice(id=0)] PID: 734990
{
  "setting_index": 0,
  "aperture_diameter_um": 4.0,
  "voltage_v": 200000.0,
  "nominal_magnification": 40000.0,
  "gaussians_across": 512,
  "gaussians": 262144,
  "samples_per_pixel_axis": 4,
  "initial_samples_per_pixel_axis": 4,
  "detector_shape": [
    256,
    256
  ],
  "field_shape": [
    1088,
    1088
  ],
  "jax_version": "0.10.2",
  "devices": "[CudaDevice(id=0)]",
  "profile_pid": 734990,
  "idle_samples_percent": [
    0,
    0,
    0,
    0
  ],
  "dense_gaussian_sample_pairs": 310311387136
}


## Construct the recorded microscope input

The production algorithm transports the source covariance through the upstream condenser, then decomposes its equivalent coherent aperture-plane wave into beamlets. Those beamlets traverse the aperture, remaining condenser, calibrated deflectors, and projector. Input construction is cold setup and normally cached for consecutive projector settings.

In [3]:
def construct_input():
    cp = effective_parameters(jnp.asarray(truth["condenser_parameters"]), voltage, spec["voltages_v"][0], vi)
    pp = at_voltage(jnp.asarray(truth["projector_parameters"]), voltage, spec["voltages_v"][0])
    centres = aperture_centres(cp, 2) * 1e-6
    beam, condenser, angular_variance = aperture_wave(
        cp, jnp.asarray(settings["condenser_controls"][SETTING_INDEX]), voltage_v=voltage,
        radius_m=diameter * 0.5e-6, edge_width_m=spec["aperture_edge_width_m"],
        centre_m=centres[APERTURE_INDEX], gaussians_across=GAUSSIANS_ACROSS)
    steered = condenser_with_steering(condenser, settings["specimen_kick"][SETTING_INDEX],
        voltage_v=voltage, gap_fractions=spec.get("deflector_gap_fractions", (0.3, 0.7)))
    projector = projector_components(pp, jnp.asarray(settings["projector_controls"][SETTING_INDEX]))
    column = tuple(replace(c, include_axial_path=False) if hasattr(c, "q") else c
                   for c in (*steered, *projector))
    return beam, column, angular_variance

beam, column, angular_variance = timed("cold_input_construction", construct_input)
assert beam.x.size == GAUSSIANS_ACROSS**2
print("Actual beamlets:", beam.x.size)
print("Column:", " → ".join(type(c).__name__ for c in column))
render = make_gaussian_renderer(shape_yx=shape, pixel_pitch_m=pitch,
    samples_per_pixel=sampling, psf_sigma_pixels=spec["psf_sigma_pixels"], padding_pixels=padding)
args = (beam, column, angular_variance, jnp.zeros(2))

cold_input_construction: 9.562265 s


Actual beamlets: 262144
Column: SigmoidAperture → DoubleDeflector → HardEdgeSolenoid → HardEdgeSolenoid → Plane → HardEdgeSolenoid → HardEdgeSolenoid → HardEdgeSolenoid → HardEdgeSolenoid → HardEdgeSolenoid → Plane


## Production renderer: compilation, warm-up, and one timed pass

Every timing waits for GPU completion. The timed pass uses the actual renderer with its normal JIT fusion; it includes aperture/lens propagation, coherent field summation, blur, and pixel integration. Input construction and the host copy are reported separately.

In [4]:
start = time.perf_counter()
compiled = render.lower(*args).compile()
timings["production_compile"] = time.perf_counter() - start
print(f"production_compile: {timings['production_compile']:.6f} s")
warm = timed("production_warmup", compiled, *args)
image = timed("production_single_pass", compiled, *args)
image_host = timed("host_copy", np.asarray, image)
assert np.isfinite(image_host).all() and image_host.sum() > 0
np.save(RUN_DIR / "image.npy", image_host)

production_compile: 5.879083 s


production_warmup: 3.980008 s


production_single_pass: 4.096223 s


host_copy: 0.000426 s


## Attribute one pass to its stages

The field and integration stages below reproduce `gaussian_image` with the same padded grid and analytic Gaussian pixel response. Each stage is compiled and warmed first; then one sequential pass is measured. The final image must agree with the production renderer.

In [5]:
sensor = Detector(z=0.0, shape=shape, pixel_size=(pitch, pitch))
grid = replace(sensor, shape=tuple(metadata["field_shape"]), pixel_size=(pitch/sampling, pitch/sampling))

@jax.jit
def propagate(source, components):
    return run_to_end(source, components, propagator=FreeSpacePropagator(include_axial_path=False))

@jax.jit
def blur_width(source, components, variance):
    matrix = column_matrix(components, source_z=jnp.ravel(jnp.asarray(source.z))[0],
                           voltage=jnp.ravel(jnp.asarray(source.voltage))[0])
    return jnp.sqrt(variance * jnp.sum(matrix[:2, 2]**2) + (spec["psf_sigma_pixels"]*pitch)**2)

@jax.jit
def field_at_detector(output):
    return evaluate_gaussians(output, grid, method="spatial", tile_beams=32)

@jax.jit
def integrate(field, blur):
    sigma = jnp.maximum(blur, 1e-30)
    def response(count):
        count_high = (count + 2*padding)*sampling
        positions = (jnp.arange(count_high) + 0.5 - count_high/2)*pitch/sampling
        edges = (jnp.arange(count + 1) - count/2)*pitch
        values = ndtr((edges[:, None] - positions[None, :])/sigma)
        return values[1:] - values[:-1]
    return response(shape[0]) @ (jnp.abs(field)**2) @ response(shape[1]).T * (pitch*pitch/sampling**2)

output = timed("stage_propagation_compile_and_warmup", propagate, beam, column)
blur = timed("stage_blur_compile_and_warmup", blur_width, beam, column, angular_variance)
field = timed("stage_field_compile_and_warmup", field_at_detector, output)
_ = timed("stage_integration_compile_and_warmup", integrate, field, blur)

output = timed("stage_propagation", propagate, beam, column)
blur = timed("stage_blur_width", blur_width, beam, column, angular_variance)
field = timed("stage_coherent_field", field_at_detector, output)
split_image = timed("stage_pixel_integration", integrate, field, blur)
split_host = np.asarray(split_image)
relative_error = float(np.max(np.abs(split_host-image_host)) / np.max(np.abs(image_host)))
np.testing.assert_allclose(split_host, image_host, rtol=1e-8, atol=1e-12*np.max(image_host))
metadata["split_vs_production_max_relative_to_peak"] = relative_error
print("Split/production maximum error relative to peak:", relative_error)

stage_propagation_compile_and_warmup: 3.769590 s


stage_blur_compile_and_warmup: 0.514620 s


stage_field_compile_and_warmup: 4.867075 s


stage_integration_compile_and_warmup: 0.333199 s


stage_propagation: 0.013242 s


stage_blur_width: 0.001004 s


stage_coherent_field: 4.433443 s


stage_pixel_integration: 0.000458 s


Split/production maximum error relative to peak: 0.0


## Did spatial binning help?

The spatial method switches to dense evaluation when `9 × largest_bin_population ≥ number_of_beamlets`, or if beam support is invalid. The dense path evaluates every beamlet at every detector sample, including beamlets with tiny aperture transmission.

In [6]:
params = _prepare_gaussian_params(output)
bins = jax.jit(lambda r, q, k: _build_spatial_bins(r, q, k, cutoff=1e-12, bins_per_axis=64))(
    params[0], params[4], params[5])
jax.block_until_ready(bins)
metadata.update({
    "spatial_dense_fallback": bool(bins.use_dense),
    "largest_bin_population": int(jnp.max(bins.counts)),
    "occupied_bins": int(jnp.count_nonzero(bins.counts)),
    "spatial_cell_size_m": float(bins.cell_size),
    "nonzero_amplitude_beamlets": int(jnp.count_nonzero(output.amplitude)),
    "amplitude_below_1e_minus12_of_max": int(jnp.count_nonzero(jnp.abs(output.amplitude) < jnp.max(jnp.abs(output.amplitude))*1e-12)),
})
for key in ["spatial_dense_fallback", "largest_bin_population", "occupied_bins", "spatial_cell_size_m",
            "nonzero_amplitude_beamlets", "amplitude_below_1e_minus12_of_max"]:
    print(f"{key}: {metadata[key]}")

spatial_dense_fallback: True
largest_bin_population: 262144
occupied_bins: 1
spatial_cell_size_m: 1.1040126909335448
nonzero_amplitude_beamlets: 262144
amplitude_below_1e_minus12_of_max: 17037


## GPU trace of one warmed production pass

This additional pass records device kernels, transfers, and host activity. Its wall time includes profiler overhead and is kept separate from the uninstrumented timing above. Trace files remain local under the printed artifact directory.

In [7]:
trace_dir = RUN_DIR / "jax_trace"
with jax.profiler.trace(trace_dir, create_perfetto_trace=True):
    with jax.profiler.TraceAnnotation("microscope_single_pass"):
        traced_image = timed("production_profiled_pass", compiled, *args)
np.testing.assert_allclose(np.asarray(traced_image), image_host, rtol=1e-12, atol=0)
print("Trace:", trace_dir)

production_profiled_pass: 4.453564 s


Trace: /Users/landers/TemGymCore/examples/generated/microscope_single_pass_profile/20260909T130542Z/jax_trace


In [8]:
monitor.terminate()
monitor.wait(timeout=10)
# pmon reports per-process SM use once per second. Ignore this notebook's own PID.
foreign_activity = []
for line in (RUN_DIR / "gpu_activity.log").read_text().splitlines():
    parts = line.split()
    if len(parts) > 4 and parts[0].isdigit() and parts[1].isdigit():
        if int(parts[1]) != os.getpid() and parts[3].isdigit() and int(parts[3]) > 0:
            foreign_activity.append(line)
metadata["other_process_activity_detected"] = foreign_activity
metadata["other_process_activity_sampling_interval_s"] = 1
metadata["timings_seconds"] = timings
(RUN_DIR / "results.json").write_text(json.dumps(metadata, indent=2) + "\n")
with (RUN_DIR / "timings.csv").open("w") as f:
    writer = csv.writer(f); writer.writerow(["measurement", "seconds"]); writer.writerows(timings.items())
stages = ["stage_propagation", "stage_blur_width", "stage_coherent_field", "stage_pixel_integration"]
total = sum(timings[k] for k in stages)
rows = "\n".join(f"| {k.removeprefix('stage_').replace('_', ' ')} | {timings[k]:.6f} | {100*timings[k]/total:.2f}% |" for k in stages)
display(Markdown(f"| Stage | Seconds | Share of split pass |\n|---|---:|---:|\n{rows}"))
display(Markdown(f"**Production single image:** {timings['production_single_pass']:.3f} s. "
    f"**Compilation:** {timings['production_compile']:.3f} s. "
    f"**Gaussians:** {metadata['gaussians']:,}. "
    f"**Spatial dense fallback:** {metadata['spatial_dense_fallback']}."))
print("Other-process GPU activity samples:", foreign_activity)
assert not foreign_activity, "GPU activity from another process was detected; repeat after it finishes."
print("Results:", RUN_DIR / "results.json")

| Stage | Seconds | Share of split pass |
|---|---:|---:|
| propagation | 0.013242 | 0.30% |
| blur width | 0.001004 | 0.02% |
| coherent field | 4.433443 | 99.67% |
| pixel integration | 0.000458 | 0.01% |

**Production single image:** 4.096 s. **Compilation:** 5.879 s. **Gaussians:** 262,144. **Spatial dense fallback:** True.

Other-process GPU activity samples: ['    0     735427     C     11      0      -      -      -      -   1070      0    python         ', '    0     735427     C     24      0      -      -      -      -   1070      0    python         ', '    0     735427     C     22      0      -      -      -      -   1070      0    python         ', '    0     735427     C     20      0      -      -      -      -   1070      0    python         ', '    0     735427     C     20      0      -      -      -      -   1070      0    python         ', '    0     735427     C     17      0      -      -      -      -   1070      0    python         ', '    0     735427     C      5      0      -      -      -      -   1070      0    python         ', '    0     735427     C     11      0      -      -      -      -   1070      0    python         ', '    0     735427     C     22      0      -      -      -      -   1070      0    python         ', '    0     735427     C     22      0      -      -   

AssertionError: GPU activity from another process was detected; repeat after it finishes.